In [0]:
spark.sql("DROP TABLE IF EXISTS carros")

DataFrame[]

In [0]:
carros = spark.read.load("/Volumes/workspace/default/dados/Carros.csv",
                        format = "CSV", sep = ";", inferSchema = True, header=True)
carros.write.mode("overwrite").saveAsTable("carros")

In [0]:
%sql
select * from carros

Consumo,Cilindros,Cilindradas,RelEixoTraseiro,Peso,Tempo,TipoMotor,Transmissao,Marchas,Carburadors,HP
21,6,160,39,262,1646,0,1,4,4,110
21,6,160,39,2875,1702,0,1,4,4,110
228,4,108,385,232,1861,1,1,4,1,93
214,6,258,308,3215,1944,1,0,3,1,110
187,8,360,315,344,1702,0,0,3,2,175
181,6,225,276,346,2022,1,0,3,1,105
143,8,360,321,357,1584,0,0,3,4,245
244,4,1467,369,319,20,1,0,4,2,62
228,4,1408,392,315,229,1,0,4,2,95
192,6,1676,392,344,183,1,0,4,4,123


In [0]:
from pyspark.sql.functions import col, when
carros_df = spark.table("carros")

# nomes da pasta
carros_df.write.format("delta").save("/Volumes/workspace/default/dados/carros/carros_delta")


Lendo tabela delta

In [0]:
delta_df = spark.read.format("delta").load("/Volumes/workspace/default/dados/carros/carros_delta")
delta_df.show()

+-------+---------+-----------+---------------+----+-----+---------+-----------+-------+-----------+---+
|Consumo|Cilindros|Cilindradas|RelEixoTraseiro|Peso|Tempo|TipoMotor|Transmissao|Marchas|Carburadors| HP|
+-------+---------+-----------+---------------+----+-----+---------+-----------+-------+-----------+---+
|     21|        6|        160|             39| 262| 1646|        0|          1|      4|          4|110|
|     21|        6|        160|             39|2875| 1702|        0|          1|      4|          4|110|
|    228|        4|        108|            385| 232| 1861|        1|          1|      4|          1| 93|
|    214|        6|        258|            308|3215| 1944|        1|          0|      3|          1|110|
|    187|        8|        360|            315| 344| 1702|        0|          0|      3|          2|175|
|    181|        6|        225|            276| 346| 2022|        1|          0|      3|          1|105|
|    143|        8|        360|            321| 357| 15

Atualizando tipo de motor

In [0]:
delta_df = delta_df.withColumn('TipoMotor', when(col('TipoMotor') == 0, 2).otherwise(col("TipoMotor")))


Grava, Carrega e exibe

In [0]:
delta_df.write.format("delta").mode('overwrite').save("/Volumes/workspace/default/dados/carros/carros_delta")
updated_delta_df = spark.read.format("delta").load("/Volumes/workspace/default/dados/carros/carros_delta")
updated_delta_df.show()

+-------+---------+-----------+---------------+----+-----+---------+-----------+-------+-----------+---+
|Consumo|Cilindros|Cilindradas|RelEixoTraseiro|Peso|Tempo|TipoMotor|Transmissao|Marchas|Carburadors| HP|
+-------+---------+-----------+---------------+----+-----+---------+-----------+-------+-----------+---+
|     21|        6|        160|             39| 262| 1646|        2|          1|      4|          4|110|
|     21|        6|        160|             39|2875| 1702|        2|          1|      4|          4|110|
|    228|        4|        108|            385| 232| 1861|        1|          1|      4|          1| 93|
|    214|        6|        258|            308|3215| 1944|        1|          0|      3|          1|110|
|    187|        8|        360|            315| 344| 1702|        2|          0|      3|          2|175|
|    181|        6|        225|            276| 346| 2022|        1|          0|      3|          1|105|
|    143|        8|        360|            321| 357| 15

Consultando versões existentes

In [0]:
%sql
SELECT version, timestamp, operation,  operationMetrics
FROM(
    DESCRIBE HISTORY delta.`/Volumes/workspace/default/dados/carros/carros_delta`
)


version,timestamp,operation,operationMetrics
1,2026-05-20T00:12:06.000Z,WRITE,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 3622, numDeletionVectorsRemoved -> 0, numOutputRows -> 32, numOutputBytes -> 3623)"
0,2026-05-20T00:03:06.000Z,WRITE,"Map(numFiles -> 1, numOutputRows -> 32, numOutputBytes -> 3622)"


Lendo versão anterior

In [0]:
version0_df = spark.read.format("delta").option("VersionAsOf", 0).load("/Volumes/workspace/default/dados/carros/carros_delta")
version0_df.show()

+-------+---------+-----------+---------------+----+-----+---------+-----------+-------+-----------+---+
|Consumo|Cilindros|Cilindradas|RelEixoTraseiro|Peso|Tempo|TipoMotor|Transmissao|Marchas|Carburadors| HP|
+-------+---------+-----------+---------------+----+-----+---------+-----------+-------+-----------+---+
|     21|        6|        160|             39| 262| 1646|        0|          1|      4|          4|110|
|     21|        6|        160|             39|2875| 1702|        0|          1|      4|          4|110|
|    228|        4|        108|            385| 232| 1861|        1|          1|      4|          1| 93|
|    214|        6|        258|            308|3215| 1944|        1|          0|      3|          1|110|
|    187|        8|        360|            315| 344| 1702|        0|          0|      3|          2|175|
|    181|        6|        225|            276| 346| 2022|        1|          0|      3|          1|105|
|    143|        8|        360|            321| 357| 15

Executando ultima versão da tabela

In [0]:
updated_delta_df.show()

+-------+---------+-----------+---------------+----+-----+---------+-----------+-------+-----------+---+
|Consumo|Cilindros|Cilindradas|RelEixoTraseiro|Peso|Tempo|TipoMotor|Transmissao|Marchas|Carburadors| HP|
+-------+---------+-----------+---------------+----+-----+---------+-----------+-------+-----------+---+
|     21|        6|        160|             39| 262| 1646|        2|          1|      4|          4|110|
|     21|        6|        160|             39|2875| 1702|        2|          1|      4|          4|110|
|    228|        4|        108|            385| 232| 1861|        1|          1|      4|          1| 93|
|    214|        6|        258|            308|3215| 1944|        1|          0|      3|          1|110|
|    187|        8|        360|            315| 344| 1702|        2|          0|      3|          2|175|
|    181|        6|        225|            276| 346| 2022|        1|          0|      3|          1|105|
|    143|        8|        360|            321| 357| 15

In [0]:
version_default = spark.read.format("delta").load("/Volumes/workspace/default/dados/carros/carros_delta")
version_default.show()

+-------+---------+-----------+---------------+----+-----+---------+-----------+-------+-----------+---+
|Consumo|Cilindros|Cilindradas|RelEixoTraseiro|Peso|Tempo|TipoMotor|Transmissao|Marchas|Carburadors| HP|
+-------+---------+-----------+---------------+----+-----+---------+-----------+-------+-----------+---+
|     21|        6|        160|             39| 262| 1646|        2|          1|      4|          4|110|
|     21|        6|        160|             39|2875| 1702|        2|          1|      4|          4|110|
|    228|        4|        108|            385| 232| 1861|        1|          1|      4|          1| 93|
|    214|        6|        258|            308|3215| 1944|        1|          0|      3|          1|110|
|    187|        8|        360|            315| 344| 1702|        2|          0|      3|          2|175|
|    181|        6|        225|            276| 346| 2022|        1|          0|      3|          1|105|
|    143|        8|        360|            321| 357| 15

Apagar arquivo delta de volume

In [0]:
dbutils.fs.rm("/Volumes/workspace/default/dados/carros/carros_delta", recurse=True)

True